# VRC07 background-dependence violins

Affinity distributions for every combinatorial state of a set of focal mutations, drawn separately per antigen, with the remaining sites pooled. White points mark group means and grey lines the medians; the mutation matrix beneath each panel gives the state of each site.

## 1. Setup

In [ ]:
import os
import colorsys
import itertools
import re
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.colors import to_rgb, to_hex
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde

%matplotlib inline

## 2. Paths and constants

In [ ]:
# ---- paths -------------------------------------------------------------
data_dir = "../data"
fig_dir = "../figures"
table_dir = "../tables"
os.makedirs(fig_dir, exist_ok=True)
os.makedirs(table_dir, exist_ok=True)
PNG_DPI = 300

plt.rcParams["svg.fonttype"] = "path"   # illustrator drops live text
plt.rcParams["pdf.fonttype"] = 42

# display name to file stem; the hyphens and the dot do not appear in any filename
NAME_TO_STEM = {"1NC9": "1NC9", "CH235.12": "CH235_12", "N49P7": "N49P7",
                "NIH45-46": "NIH4546", "VRC-PG20": "VRCPG20", "VRC01": "VRC01",
                "VRC07": "VRC07", "VRC13": "VRC13", "VRC16": "VRC16"}


def save_figure(fig, stem):
    """Write png, pdf and svg, then check the svg has no live text."""
    path = f"{fig_dir}/{stem}"
    fig.savefig(f"{path}.png", dpi=PNG_DPI, bbox_inches="tight")
    fig.savefig(f"{path}.pdf", bbox_inches="tight")
    fig.savefig(f"{path}.svg", bbox_inches="tight")
    assert open(f"{path}.svg").read().count("<text") == 0, "live text in svg, check svg.fonttype"


# ---- constants (inlined from the workflow's config.py) -----------------
ANTIBODIES = ["1NC9", "CH235.12", "VRC01", "VRC13", "VRC16",
              "VRC07", "N49P7", "NIH45-46", "VRC-PG20"]
ANTIGENS = ("CH505TF", "SF162")

GERMLINE_TOKEN = "G"        # site_seq character meaning "germline at this site"
VALUE_COL_RAW = "nlog10_Kd_raw"
FLOOR_CENSOR_VALUE = 6.0    # assay floor; values at exactly this are censored

GERMLINE = GERMLINE_TOKEN
VALUE_COL = VALUE_COL_RAW

## 3. Data loaders

In [ ]:
def load_raw_binding_data():
    """Pooled genotype table assembled from the per-antibody kds files.

    One row per antibody, antigen and genotype, replicate-averaged, with the
    G/M token string in site_seq. Reconstructs what the old pooled
    final_cleaned_all_antibody_binding_data.csv provided.
    """
    frames = []
    for ab in NAME_TO_STEM:
        d = pd.read_csv(f"{data_dir}/{NAME_TO_STEM[ab]}_kds.csv")
        d["antibody"] = ab
        d["site_seq"] = d[[f"site_{i}" for i in range(10)]].agg("".join, axis=1)
        frames.append(d)
    df = pd.concat(frames, ignore_index=True)
    df = (df.groupby(["antibody", "antigen", "site_seq", "id"], as_index=False)["nlog10_Kd"]
            .mean()
            .rename(columns={"nlog10_Kd": "nlog10_Kd_raw"}))
    print(f"Loaded raw binding data: {df.shape} from {len(NAME_TO_STEM)} kds files")
    return df


def find_metadata_file(antibody):
    """Path to one antibody's site metadata."""
    return f"{data_dir}/{NAME_TO_STEM[antibody]}_meta.csv"


def kabat_to_site_index(antibody):
    # {'L45P': 3, ...} -- maps a mutation label to its position in site_seq.
    meta = pd.read_csv(find_metadata_file(antibody))
    meta.columns = meta.columns.str.strip()
    mapping = {}
    for _, row in meta.iterrows():
        site_index = int(str(row["site"]).replace("site_", ""))
        mapping[f"{row['germline_aa']}{row['kabat_position']}{row['mature_aa']}"] = site_index
    return mapping

## 4. Shared helpers (inlined)

In [ ]:
def lighten(rgb, sat_mult=0.55, val_boost=1.25):
    """Same hue, pushed toward lighter/less-saturated -- the inverse of
    darken(), used to derive the 'absent/germline' half FROM a colorblind-
    safe base hue instead of starting pastel and darkening for 'mature'."""
    h, s, v = colorsys.rgb_to_hsv(*rgb[:3])
    s = max(0.0, s * sat_mult)
    v = min(1.0, v * val_boost)
    return colorsys.hsv_to_rgb(h, s, v)
def darken(rgb, sat_boost=1.8, val_mult=0.6):
    """Same hue, pushed toward darker/more saturated -- used for the
    'present/mature' half so it visually reads as bolder than the pastel
    'absent/germline' base color."""
    h, s, v = colorsys.rgb_to_hsv(*rgb[:3])
    s = min(1.0, s * sat_boost + 0.1)
    v = max(0.0, v * val_mult)
    return colorsys.hsv_to_rgb(h, s, v)

## 5. Figure code

In [ ]:
matplotlib.use("Agg")


plt.rcParams.update({"font.size": 10})

ORDER0_COLOR = "#6B5894"  # purple -- germline
ORDER1_COLORS = ["#8EB4FB", "#002792", "#386A4F"]  # light blue, dark blue, green
ORDER2_COLORS = ["#CFB454", "#B96040", "#CC688B"]  # yellow, orange, light pink
ORDER3_COLOR = "#741345"  # magenta -- all three

_LIGHTEN_T1 = dict(sat_mult=0.70, val_boost=1.15)  # mild
_LIGHTEN_T2 = dict(sat_mult=0.42, val_boost=1.30)  # stronger, still single-pass


def _lighten_hex(hexcolor, tier=_LIGHTEN_T1):
    return to_hex(lighten(to_rgb(hexcolor), **tier))


_HUES = list(ORDER1_COLORS) + list(ORDER2_COLORS)  # h0,h1,h2 (order1) ; h3,h4,h5 (order2)

ORDER1_COLORS_4 = [_HUES[0], _HUES[1], _HUES[2], _lighten_hex(_HUES[0], _LIGHTEN_T1)]
ORDER2_COLORS_4 = [_HUES[3], _HUES[4], _HUES[5],
                    _lighten_hex(_HUES[3], _LIGHTEN_T1), _lighten_hex(_HUES[4], _LIGHTEN_T1),
                    _lighten_hex(_HUES[5], _LIGHTEN_T1)]
ORDER3_COLORS_4 = [_lighten_hex(_HUES[1], _LIGHTEN_T1), _lighten_hex(_HUES[2], _LIGHTEN_T1),
                    _lighten_hex(_HUES[3], _LIGHTEN_T2), _lighten_hex(_HUES[4], _LIGHTEN_T2)]
ORDER4_COLOR = ORDER3_COLOR  # magenta -- all four mutant (same bookend as the 3-mutation figure's triple)


def _hue_gradient_hex(light_hex, dark_hex, k):
    if k == 1:
        return [light_hex]
    lr, lg, lb = to_rgb(light_hex)
    dr, dg, db = to_rgb(dark_hex)
    return [to_hex((lr + (dr - lr) * i / (k - 1),
                     lg + (dg - lg) * i / (k - 1),
                     lb + (db - lb) * i / (k - 1)))
            for i in range(k)]


def _mutfamily_palette_4():
    def _darken_hex(hexcolor, sat_boost, val_mult):
        return to_hex(darken(to_rgb(hexcolor), sat_boost=sat_boost, val_mult=val_mult))

    def _lighten_hex_local(hexcolor, sat_mult, val_boost):
        return to_hex(lighten(to_rgb(hexcolor), sat_mult=sat_mult, val_boost=val_boost))

    # (light endpoint, dark endpoint) per mutation index, reusing core hues
    # directly wherever the core palette already has 2 for that role.
    endpoints = [
        (ORDER1_COLORS[0], _darken_hex(ORDER1_COLORS[1], sat_boost=1.55, val_mult=0.62)),  # L45P: light blue -> near-black blue
        (_lighten_hex_local(ORDER1_COLORS[2], sat_mult=0.6, val_boost=1.35),
         _darken_hex(ORDER1_COLORS[2], sat_boost=1.7, val_mult=0.55)),                      # N52K: pale green -> deep green
        (ORDER2_COLORS[0], ORDER2_COLORS[1]),                                               # S54G: yellow -> orange
        (ORDER2_COLORS[2], ORDER2_COLORS[2]),                                               # I51L: light pink (always 1 member)
    ]

    by_anchor = {0: [], 1: [], 2: [], 3: []}
    for r in range(1, 4):
        for combo in itertools.combinations(range(4), r):
            by_anchor[combo[0]].append(combo)

    combo_color = {}
    for anchor, combos in by_anchor.items():
        light_hex, dark_hex = endpoints[anchor]
        for combo, color in zip(combos, _hue_gradient_hex(light_hex, dark_hex, len(combos))):
            combo_color[combo] = color

    states = [()]
    colors = [ORDER0_COLOR]
    for r in range(1, 4):
        for combo in itertools.combinations(range(4), r):
            states.append(combo)
            colors.append(combo_color[combo])
    states.append((0, 1, 2, 3))
    colors.append(ORDER3_COLOR)
    return states, colors


def three_way_order_violin_figure(df, antibody, muts, out_name, restrict_germline_at=None,
                                   restrict_mutant_at=None, restrict_any_mutant_at=None,
                                   antigens=None, figsize=None, min_n=4, title_note=None,
                                   show_title=True, show_panel_titles=True, wspace=None,
                                   dashed_lines=True, capitalize_ylabel=False):

    assert len(muts) == 3, "this figure is specifically for a 3-mutation trio"
    assert not (set(restrict_germline_at or []) & set(restrict_mutant_at or [])), \
        "a site can't be restricted to both germline and mutant at once"
    antigens = antigens or list(ANTIGENS)
    k2i = kabat_to_site_index(antibody)
    idxs = [k2i[m] for m in muts]

    states = [()]
    colors = [ORDER0_COLOR]
    pairs = list(itertools.combinations(range(3), 2))
    for r, palette in ((1, ORDER1_COLORS), (2, ORDER2_COLORS)):
        for combo in itertools.combinations(range(3), r):
            states.append(combo)
            idx = combo[0] if r == 1 else pairs.index(combo)
            colors.append(palette[idx])
    states.append((0, 1, 2))
    colors.append(ORDER3_COLOR)
    group_sizes = [1, 3, 3, 1]  # germline | singles | doubles | triple

    n = len(states)

    def state_mask(seq, state):
        for j, idx in enumerate(idxs):
            mutated = seq[idx] != GERMLINE
            if (j in state) != mutated:
                return False
        return True

    # First pass: filter + bin each antigen's data, and track the global
    # y-range across every antigen/state so all panels can share one y-axis.
    per_antigen = []
    global_vals = []
    for antigen in antigens:
        d = df[(df.antibody == antibody) & (df.antigen == antigen)].dropna(subset=[VALUE_COL]).copy()
        for m in (restrict_germline_at or []):
            gi = k2i[m]
            d = d[d["site_seq"].str[gi] == GERMLINE]
        for m in (restrict_mutant_at or []):
            mi = k2i[m]
            d = d[d["site_seq"].str[mi] != GERMLINE]
        if restrict_any_mutant_at:
            any_mut_idxs = [k2i[m] for m in restrict_any_mutant_at]
            is_any_mut = d["site_seq"].apply(lambda s: any(s[i] != GERMLINE for i in any_mut_idxs))
            d = d[is_any_mut]

        data, ns = [], []
        for state in states:
            vals = d.loc[d["site_seq"].apply(lambda s: state_mask(s, state)), VALUE_COL].values
            data.append(vals)
            ns.append(len(vals))
            if len(vals) >= min_n:
                global_vals.append(vals)
        per_antigen.append((antigen, data, ns))

    global_vals = np.concatenate(global_vals) if global_vals else np.array([FLOOR_CENSOR_VALUE])
    y_lo = min(FLOOR_CENSOR_VALUE, float(global_vals.min()))
    y_hi = float(global_vals.max())
    pad = (y_hi - y_lo) * 0.05
    y_lo, y_hi = y_lo - pad, y_hi + pad

    if wspace is None:
        wspace = 0.18 if len(antigens) > 1 else 0
    figsize = figsize or (4.4 * len(antigens) + 1, 6.6)
    fig = plt.figure(figsize=figsize, dpi=170)
    gs = fig.add_gridspec(2, len(antigens), height_ratios=[3.3, 1], hspace=0.08,
                           wspace=wspace)

    shared_ax = None
    for col, (antigen, data, ns) in enumerate(per_antigen):
        ax = fig.add_subplot(gs[0, col], sharey=shared_ax)
        shared_ax = shared_ax or ax
        ax_dot = fig.add_subplot(gs[1, col], sharex=ax)

        xs = list(range(n))
        for x, vals, color in zip(xs, data, colors):
            if len(vals) < min_n:
                continue
            parts = ax.violinplot([vals], positions=[x], widths=0.75,
                                   showmeans=False, showmedians=False, showextrema=False)
            for body in parts["bodies"]:
                body.set_facecolor(color)
                body.set_alpha(0.85)
                body.set_edgecolor("black")
                body.set_linewidth(0.8)
            # median line + mean dot, both shown -- matches the 2026-07-30
            # split-violin restyle (see feedback_split_violin_gating_convention)
            median = np.median(vals)
            ax.plot([x - 0.34, x + 0.34], [median, median], color="#555555", linewidth=1.1, zorder=6)
            ax.scatter([x], [np.mean(vals)], s=34, color="white", edgecolor="black",
                       linewidths=1.1, zorder=7)

        # separators between order-groups (after cumulative group sizes) on
        # BOTH the violin and dot-matrix axes -- renamed loop var (gsz, not
        # gs) since `gs` is this figure's GridSpec object. Dashed by
        # default; omitted entirely (not just solid) when dashed_lines=False.
        if dashed_lines:
            cum = 0
            for gsz in group_sizes[:-1]:
                cum += gsz
                ax.axvline(cum - 0.5, color="0.6", linestyle="--", linewidth=1, zorder=1)
                ax_dot.axvline(cum - 0.5, color="0.85", linestyle="--", linewidth=0.7, zorder=0)

        # floor-censoring reference line + "<=6" tick relabel, same touch
        # added to the raw split-violin panels in the same restyle pass.
        # ylim is the GLOBAL range computed above (shared via sharey=), set
        # explicitly here so the tick locator below picks ticks for it.
        ax.axhline(FLOOR_CENSOR_VALUE, color="#bbbbbb", linestyle="-", linewidth=0.8, zorder=2)
        ax.set_ylim(y_lo, y_hi)
        yticks = [t for t in ax.get_yticks() if y_lo - 1e-9 <= t <= y_hi + 1e-9]
        if yticks:
            ax.set_yticks(yticks)
            ax.set_yticklabels([f"≤{FLOOR_CENSOR_VALUE:g}" if abs(t - FLOOR_CENSOR_VALUE) < 1e-6
                                 else f"{t:g}" for t in yticks])
        ax.set_xticks([])
        ax.set_xlim(-0.7, n - 0.3)
        if show_panel_titles:
            ax.set_title(antigen, fontsize=12, fontweight="bold")
        # No gridlines -- matches the reference script's split-violin
        # panel C (no ax.grid call there) and draw_violin_panel in
        # plot_final_delta_violins_unified.py, streamlined to match.
        spines_to_hide = ["top", "right", "bottom"]
        for sp in spines_to_hide:
            ax.spines[sp].set_visible(False)

        # Dot-matrix row per trio mutation -- filled dark (#222222) =
        # present, filled light gray (#e2e2e2) = absent, same encoding as
        # draw_dot_matrix_panel in plot_final_delta_violins_unified.py.
        # n= labels sit at a fixed y below the rows rather than in the
        # x-tick text, matching that same convention.
        n_rows = len(muts)
        row_ys = list(range(n_rows - 1, -1, -1))
        for x, state in zip(xs, states):
            present_js = set(state)
            for j, ry in zip(range(n_rows), row_ys):
                present = j in present_js
                ax_dot.scatter(x, ry, s=48, color="#222222" if present else "#e2e2e2",
                               edgecolor="none", zorder=5, clip_on=False)
        for mut, ry in zip(muts, row_ys):
            ax_dot.annotate(mut, xy=(0, ry), xycoords=("axes fraction", "data"),
                            xytext=(-6, 0), textcoords="offset points",
                            ha="right", va="center", fontsize=7.5, clip_on=False)

        n_label_y = min(row_ys) - 1.0
        ax_dot.text(-0.85, -1.0, "n=", ha="right", va="top", fontsize=6, color="#262626")
        for x, ni in zip(xs, ns):
            ax_dot.text(x, n_label_y, f"{ni}", ha="center", va="top",
                        fontsize=6, color="#262626", clip_on=False)

        ax_dot.set_xlim(-0.7, n - 0.3)
        ax_dot.set_ylim(n_label_y - 0.6, max(row_ys) + 0.6)
        ax_dot.set_xticks([])
        ax_dot.set_yticks([])
        for sp in ax_dot.spines.values():
            sp.set_visible(False)

        ylabel_text = "Binding Affinity" if capitalize_ylabel else "binding affinity"
        ax.set_ylabel(f"{antigen}\n" + ylabel_text + r" ($-\log_{10}K_D$)", fontsize=11)

    if show_title:
        title = f"{antibody}: 3rd-order interaction\nMutations {', '.join(muts)}"
        if title_note:
            title += f" ({title_note})"
        fig.suptitle(title, fontsize=13, fontweight="bold", y=1.03)
    plt.tight_layout()

    save_figure(fig, Path(out_name).stem)
    plt.close(fig)


def four_way_order_violin_figure(df, antibody, muts, out_name, antigens=None,
                                  figsize=None, min_n=4, restrict_germline_at=None,
                                  title_note=None, palette_override=None,
                                  show_title=True, show_panel_titles=True):

    assert len(muts) == 4
    antigens = antigens or list(ANTIGENS)
    k2i = kabat_to_site_index(antibody)
    idxs = [k2i[m] for m in muts]

    if palette_override is not None:
        states, colors = palette_override
    else:
        states = [()]
        colors = [ORDER0_COLOR]
        for r, palette in ((1, ORDER1_COLORS_4), (2, ORDER2_COLORS_4), (3, ORDER3_COLORS_4)):
            for combo, color in zip(itertools.combinations(range(4), r), palette):
                states.append(combo)
                colors.append(color)
        states.append((0, 1, 2, 3))
        colors.append(ORDER4_COLOR)
    group_sizes = [1, 4, 6, 4, 1]  # germline | singles | doubles | triples | quadruple
    n = len(states)

    def state_mask(seq, state):
        for j, idx in enumerate(idxs):
            mutated = seq[idx] != GERMLINE
            if (j in state) != mutated:
                return False
        return True

    per_antigen = []
    global_vals = []
    for antigen in antigens:
        d = df[(df.antibody == antibody) & (df.antigen == antigen)].dropna(subset=[VALUE_COL]).copy()
        for m in (restrict_germline_at or []):
            gi = k2i[m]
            d = d[d["site_seq"].str[gi] == GERMLINE]
        data, ns = [], []
        for state in states:
            vals = d.loc[d["site_seq"].apply(lambda s: state_mask(s, state)), VALUE_COL].values
            data.append(vals)
            ns.append(len(vals))
            if len(vals) >= min_n:
                global_vals.append(vals)
        per_antigen.append((antigen, data, ns))

    global_vals = np.concatenate(global_vals) if global_vals else np.array([FLOOR_CENSOR_VALUE])
    y_lo = min(FLOOR_CENSOR_VALUE, float(global_vals.min()))
    y_hi = float(global_vals.max())
    pad = (y_hi - y_lo) * 0.05
    y_lo, y_hi = y_lo - pad, y_hi + pad

    figsize = figsize or (0.62 * n * len(antigens) + 2, 6.8)
    fig = plt.figure(figsize=figsize, dpi=170)
    gs = fig.add_gridspec(2, len(antigens), height_ratios=[3.3, 1], hspace=0.08,
                           wspace=0.18 if len(antigens) > 1 else 0)

    shared_ax = None
    for col, (antigen, data, ns) in enumerate(per_antigen):
        ax = fig.add_subplot(gs[0, col], sharey=shared_ax)
        shared_ax = shared_ax or ax
        ax_dot = fig.add_subplot(gs[1, col], sharex=ax)

        xs = list(range(n))
        for x, vals, color in zip(xs, data, colors):
            if len(vals) < min_n:
                continue
            parts = ax.violinplot([vals], positions=[x], widths=0.75,
                                   showmeans=False, showmedians=False, showextrema=False)
            for body in parts["bodies"]:
                body.set_facecolor(color)
                body.set_alpha(0.85)
                body.set_edgecolor("black")
                body.set_linewidth(0.7)
            median = np.median(vals)
            ax.plot([x - 0.34, x + 0.34], [median, median], color="#555555", linewidth=1.0, zorder=6)
            ax.scatter([x], [np.mean(vals)], s=28, color="white", edgecolor="black",
                       linewidths=0.9, zorder=7)

        cum = 0
        for gsz in group_sizes[:-1]:
            cum += gsz
            ax.axvline(cum - 0.5, color="0.6", linestyle="--", linewidth=1, zorder=1)
            ax_dot.axvline(cum - 0.5, color="0.85", linestyle="--", linewidth=0.7, zorder=0)

        ax.axhline(FLOOR_CENSOR_VALUE, color="#bbbbbb", linestyle="--", linewidth=0.8, zorder=2)
        ax.set_ylim(y_lo, y_hi)
        yticks = [t for t in ax.get_yticks() if y_lo - 1e-9 <= t <= y_hi + 1e-9]
        if yticks:
            ax.set_yticks(yticks)
            ax.set_yticklabels([f"≤{FLOOR_CENSOR_VALUE:g}" if abs(t - FLOOR_CENSOR_VALUE) < 1e-6
                                 else f"{t:g}" for t in yticks])
        ax.set_xticks([])
        ax.set_xlim(-0.7, n - 0.3)
        if show_panel_titles:
            ax.set_title(antigen, fontsize=12, fontweight="bold")
        spines_to_hide = ["top", "right", "bottom"]
        for sp in spines_to_hide:
            ax.spines[sp].set_visible(False)

        n_rows = len(muts)
        row_ys = list(range(n_rows - 1, -1, -1))
        for x, state in zip(xs, states):
            present_js = set(state)
            for j, ry in zip(range(n_rows), row_ys):
                present = j in present_js
                ax_dot.scatter(x, ry, s=42, color="#222222" if present else "#e2e2e2",
                               edgecolor="none", zorder=5, clip_on=False)
        for mut, ry in zip(muts, row_ys):
            ax_dot.annotate(mut, xy=(0, ry), xycoords=("axes fraction", "data"),
                            xytext=(-6, 0), textcoords="offset points",
                            ha="right", va="center", fontsize=7.5, clip_on=False)

        n_label_y = min(row_ys) - 1.0
        ax_dot.text(-0.85, -1.0, "n=", ha="right", va="top", fontsize=6, color="#262626")
        for x, ni in zip(xs, ns):
            ax_dot.text(x, n_label_y, f"{ni}", ha="center", va="top",
                        fontsize=6, color="#262626", clip_on=False)

        ax_dot.set_xlim(-0.7, n - 0.3)
        ax_dot.set_ylim(n_label_y - 0.6, max(row_ys) + 0.6)
        ax_dot.set_xticks([])
        ax_dot.set_yticks([])
        for sp in ax_dot.spines.values():
            sp.set_visible(False)

        ax.set_ylabel(f"{antigen}\n" + r"binding affinity ($-\log_{10}K_D$)", fontsize=11)

    if show_title:
        title = f"{antibody}: 4th-order interaction\nMutations {', '.join(muts)}"
        if title_note:
            title += f" ({title_note})"
        fig.suptitle(title, fontsize=13, fontweight="bold", y=1.03)
    plt.tight_layout()

    save_figure(fig, Path(out_name).stem)
    plt.close(fig)


def _draw_block_bracket(ax, x0, x1, label, y=1.02, tick=0.014):
    """Small horizontal '[...]' bracket + centered label spanning [x0,x1] in
    data-x / axes-fraction-y, used to mark one background block's column
    range in mega_background_dependency_figure. Local helper (not shared
    with plot_vrc07_3way_exploration.py's draw_bracket) since this file has
    no other bracket use yet."""
    import matplotlib.transforms as mtransforms
    trans = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)
    ax.plot([x0, x0, x1, x1], [y - tick, y, y, y - tick], color="black",
             linewidth=1, transform=trans, clip_on=False)
    ax.text((x0 + x1) / 2, y + 0.012, label, transform=trans, ha="center",
            va="bottom", fontsize=8.5, fontweight="bold", clip_on=False)


def mega_background_dependency_y_range(df, antibody, trio_muts, bg_muts, antigens=None,
                                        restrict_germline_at=None, min_n=4, bg_joint=False,
                                        drop_or_gate=False):

    assert len(trio_muts) in (2, 3)
    assert len(bg_muts) in (1, 2)
    antigens = antigens or list(ANTIGENS)
    k2i = kabat_to_site_index(antibody)
    idxs = [k2i[m] for m in trio_muts]
    bg_idxs = [k2i[m] for m in bg_muts]

    if len(trio_muts) == 2:
        states = [(), (0,), (1,), (0, 1)]
    else:
        states = [()]
        pairs = list(itertools.combinations(range(3), 2))
        for r in (1, 2):
            for combo in itertools.combinations(range(3), r):
                states.append(combo)
        states.append((0, 1, 2))

    def state_mask(seq, state):
        for j, idx in enumerate(idxs):
            mutated = seq[idx] != GERMLINE
            if (j in state) != mutated:
                return False
        return True

    if len(bg_muts) == 1:
        bidx1 = bg_idxs[0]
        bg_masks = [lambda s: s[bidx1] == GERMLINE, lambda s: s[bidx1] != GERMLINE]
    elif bg_joint:
        bidx1, bidx2 = bg_idxs
        bg_masks = [
            lambda s: s[bidx1] == GERMLINE and s[bidx2] == GERMLINE,
            lambda s: s[bidx1] != GERMLINE and s[bidx2] != GERMLINE,
        ]
    else:
        bidx1, bidx2 = bg_idxs
        bg_masks = [
            lambda s: s[bidx1] == GERMLINE and s[bidx2] == GERMLINE,
            lambda s: s[bidx1] != GERMLINE and s[bidx2] == GERMLINE,
            lambda s: s[bidx1] == GERMLINE and s[bidx2] != GERMLINE,
        ]
        if not drop_or_gate:
            bg_masks.append(lambda s: s[bidx1] != GERMLINE or s[bidx2] != GERMLINE)
        bg_masks.append(lambda s: s[bidx1] != GERMLINE and s[bidx2] != GERMLINE)

    global_vals = []
    for antigen in antigens:
        d = df[(df.antibody == antibody) & (df.antigen == antigen)].dropna(subset=[VALUE_COL]).copy()
        for m in (restrict_germline_at or []):
            gi = k2i[m]
            d = d[d["site_seq"].str[gi] == GERMLINE]
        for bg_mask_fn in bg_masks:
            dd = d[d["site_seq"].apply(bg_mask_fn)]
            for state in states:
                vals = dd.loc[dd["site_seq"].apply(lambda s: state_mask(s, state)), VALUE_COL].values
                if len(vals) >= min_n:
                    global_vals.append(vals)

    global_vals = np.concatenate(global_vals) if global_vals else np.array([FLOOR_CENSOR_VALUE])
    y_lo = min(FLOOR_CENSOR_VALUE, float(global_vals.min()))
    y_hi = float(global_vals.max())
    pad = (y_hi - y_lo) * 0.05
    return y_lo - pad, y_hi + pad


def mega_background_dependency_figure(df, antibody, trio_muts, bg_muts, out_name,
                                       antigens=None, figsize=None, min_n=4, panel_order=None,
                                       restrict_germline_at=None, bg_joint=False,
                                       drop_or_gate=False, y_range=None,
                                       show_title=True, show_panel_titles=True):

    assert len(trio_muts) in (2, 3)
    assert len(bg_muts) in (1, 2)
    assert not (bg_joint and len(bg_muts) != 2), "bg_joint only applies to 2-site bg_muts"
    panel_order = panel_order or (list(trio_muts) + list(bg_muts))
    assert set(panel_order) == set(trio_muts) | set(bg_muts)
    antigens = antigens or list(ANTIGENS)
    k2i = kabat_to_site_index(antibody)
    idxs = [k2i[m] for m in trio_muts]
    bg_idxs = [k2i[m] for m in bg_muts]

    if len(trio_muts) == 2:
        # germline | single(0) | single(1) | double -- "double" reuses the
        # ORDER3_COLOR bookend since it's the terminal/highest-order state
        # for a 2-mutation core, same role that color plays for a 3-mutation
        # trio's full triple.
        states = [(), (0,), (1,), (0, 1)]
        colors = [ORDER0_COLOR, ORDER1_COLORS[0], ORDER1_COLORS[1], ORDER3_COLOR]
        trio_group_sizes = [1, 2, 1]  # germline | 2 singles | double
    else:
        states = [()]
        colors = [ORDER0_COLOR]
        pairs = list(itertools.combinations(range(3), 2))
        for r, palette in ((1, ORDER1_COLORS), (2, ORDER2_COLORS)):
            for combo in itertools.combinations(range(3), r):
                states.append(combo)
                idx = combo[0] if r == 1 else pairs.index(combo)
                colors.append(palette[idx])
        states.append((0, 1, 2))
        colors.append(ORDER3_COLOR)
        trio_group_sizes = [1, 3, 3, 1]  # germline | singles | doubles | triple
    n_trio = len(states)

    def state_mask(seq, state):
        for j, idx in enumerate(idxs):
            mutated = seq[idx] != GERMLINE
            if (j in state) != mutated:
                return False
        return True

    # label, row-filter mask, and the FIXED per-bg_mut dot value for each
    # block -- "germ"/"mut"/"mixed" ("mixed" only for the 2-site OR-gate
    # block, where each site's own state genuinely varies row-to-row
    # within the block).
    if len(bg_muts) == 1:
        b1 = bg_muts[0]
        bidx1 = bg_idxs[0]
        bg_blocks = [
            (f"-{b1}", lambda s: s[bidx1] == GERMLINE, {b1: "germ"}),
            (f"+{b1}", lambda s: s[bidx1] != GERMLINE, {b1: "mut"}),
        ]
    elif bg_joint:
        b1, b2 = bg_muts
        bidx1, bidx2 = bg_idxs
        bg_blocks = [
            (f"-{b1} AND -{b2}", lambda s: s[bidx1] == GERMLINE and s[bidx2] == GERMLINE,
             {b1: "germ", b2: "germ"}),
            (f"+{b1} AND +{b2}", lambda s: s[bidx1] != GERMLINE and s[bidx2] != GERMLINE,
             {b1: "mut", b2: "mut"}),
        ]
    else:
        b1, b2 = bg_muts
        bidx1, bidx2 = bg_idxs
        bg_blocks = [
            (f"-{b1} AND -{b2}", lambda s: s[bidx1] == GERMLINE and s[bidx2] == GERMLINE,
             {b1: "germ", b2: "germ"}),
            (f"+{b1} AND -{b2}", lambda s: s[bidx1] != GERMLINE and s[bidx2] == GERMLINE,
             {b1: "mut", b2: "germ"}),
            (f"-{b1} AND +{b2}", lambda s: s[bidx1] == GERMLINE and s[bidx2] != GERMLINE,
             {b1: "germ", b2: "mut"}),
        ]
        if not drop_or_gate:
            bg_blocks.append(
                (f"+{b1} OR +{b2}", lambda s: s[bidx1] != GERMLINE or s[bidx2] != GERMLINE,
                 {b1: "mixed", b2: "mixed"}))
        bg_blocks.append(
            (f"+{b1} AND +{b2}", lambda s: s[bidx1] != GERMLINE and s[bidx2] != GERMLINE,
             {b1: "mut", b2: "mut"}))
    n_blocks = len(bg_blocks)
    total_cols = n_blocks * n_trio

    # First pass: filter + bin every antigen x block x trio-state cell, and
    # track the global y-range so both antigen panels share one y-axis.
    per_antigen = []
    global_vals = []
    for antigen in antigens:
        d = df[(df.antibody == antibody) & (df.antigen == antigen)].dropna(subset=[VALUE_COL]).copy()
        for m in (restrict_germline_at or []):
            gi = k2i[m]
            d = d[d["site_seq"].str[gi] == GERMLINE]
        block_data = []
        for _, bg_mask_fn, _ in bg_blocks:
            dd = d[d["site_seq"].apply(bg_mask_fn)]
            data, ns = [], []
            for state in states:
                vals = dd.loc[dd["site_seq"].apply(lambda s: state_mask(s, state)), VALUE_COL].values
                data.append(vals)
                ns.append(len(vals))
                if len(vals) >= min_n:
                    global_vals.append(vals)
            block_data.append((data, ns))
        per_antigen.append((antigen, block_data))

    if y_range is not None:
        y_lo, y_hi = y_range
    else:
        global_vals = np.concatenate(global_vals) if global_vals else np.array([FLOOR_CENSOR_VALUE])
        y_lo = min(FLOOR_CENSOR_VALUE, float(global_vals.min()))
        y_hi = float(global_vals.max())
        pad = (y_hi - y_lo) * 0.05
        y_lo, y_hi = y_lo - pad, y_hi + pad

    figsize = figsize or (0.62 * total_cols * len(antigens) + 2, 8.4)
    fig = plt.figure(figsize=figsize, dpi=150)
    gs = fig.add_gridspec(2, len(antigens), height_ratios=[3.3, 1.5], hspace=0.1,
                           wspace=0.1 if len(antigens) > 1 else 0)

    shared_ax = None
    for col, (antigen, block_data) in enumerate(per_antigen):
        ax = fig.add_subplot(gs[0, col], sharey=shared_ax)
        shared_ax = shared_ax or ax
        ax_dot = fig.add_subplot(gs[1, col], sharex=ax)

        xs = list(range(total_cols))
        for b, (data, ns) in enumerate(block_data):
            base = b * n_trio
            for local_x, vals, color in zip(range(n_trio), data, colors):
                x = base + local_x
                if len(vals) < min_n:
                    continue
                parts = ax.violinplot([vals], positions=[x], widths=0.75,
                                       showmeans=False, showmedians=False, showextrema=False)
                for body in parts["bodies"]:
                    body.set_facecolor(color)
                    body.set_alpha(0.85)
                    body.set_edgecolor("black")
                    body.set_linewidth(0.7)
                median = np.median(vals)
                ax.plot([x - 0.34, x + 0.34], [median, median], color="#555555", linewidth=1.0, zorder=6)
                ax.scatter([x], [np.mean(vals)], s=26, color="white", edgecolor="black",
                           linewidths=0.9, zorder=7)

            # thin separators between order-sub-groups within this block
            cum = 0
            for gsz in trio_group_sizes[:-1]:
                cum += gsz
                sep_x = base + cum - 0.5
                ax.axvline(sep_x, color="0.75", linestyle="--", linewidth=0.7, zorder=1)
                ax_dot.axvline(sep_x, color="0.9", linestyle="--", linewidth=0.5, zorder=0)

        # thick separators + bracket labels between BACKGROUND blocks
        for b, (label, _, _) in enumerate(bg_blocks):
            x0, x1 = b * n_trio - 0.5, (b + 1) * n_trio - 0.5
            _draw_block_bracket(ax, x0 + 0.05, x1 - 0.05, label)
            if b > 0:
                ax.axvline(x0, color="0.25", linestyle="-", linewidth=1.3, zorder=2)
                ax_dot.axvline(x0, color="0.25", linestyle="-", linewidth=1.0, zorder=2)

        ax.axhline(FLOOR_CENSOR_VALUE, color="#bbbbbb", linestyle="--", linewidth=0.8, zorder=2)
        ax.set_ylim(y_lo, y_hi)
        yticks = [t for t in ax.get_yticks() if y_lo - 1e-9 <= t <= y_hi + 1e-9]
        if yticks:
            ax.set_yticks(yticks)
            ax.set_yticklabels([f"≤{FLOOR_CENSOR_VALUE:g}" if abs(t - FLOOR_CENSOR_VALUE) < 1e-6
                                 else f"{t:g}" for t in yticks])
        ax.set_xticks([])
        ax.set_xlim(-0.7, total_cols - 0.3)
        if show_panel_titles and len(antigens) > 1:
            ax.set_title(antigen, fontsize=13, fontweight="bold", pad=48)
        spines_to_hide = ["top", "right", "bottom"]
        for sp in spines_to_hide:
            ax.spines[sp].set_visible(False)

        n_rows = len(panel_order)
        row_ys = list(range(n_rows - 1, -1, -1))
        row_y_of = dict(zip(panel_order, row_ys))
        MUT_DARK, MUT_LIGHT = "#222222", "#e2e2e2"
        for b, ((_, ns), (_, _, bg_state)) in enumerate(zip(block_data, bg_blocks)):
            base = b * n_trio
            for local_x, state in zip(range(n_trio), states):
                x = base + local_x
                present_js = set(state)
                for j, mut in enumerate(trio_muts):
                    present = j in present_js
                    ax_dot.scatter(x, row_y_of[mut], s=36,
                                   color=MUT_DARK if present else MUT_LIGHT,
                                   edgecolor="none", zorder=5, clip_on=False)
                for mut in bg_muts:
                    ry = row_y_of[mut]
                    st = bg_state[mut]
                    if st == "mixed":
                        ax_dot.plot([x], [ry], marker="o", markersize=6, linestyle="none",
                                    markerfacecolor=MUT_DARK, markerfacecoloralt=MUT_LIGHT,
                                    fillstyle="left", markeredgewidth=0, zorder=5, clip_on=False)
                    else:
                        ax_dot.scatter(x, ry, s=36,
                                       color=MUT_DARK if st == "mut" else MUT_LIGHT,
                                       edgecolor="none", zorder=5, clip_on=False)
        for mut in panel_order:
            ax_dot.annotate(mut, xy=(0, row_y_of[mut]), xycoords=("axes fraction", "data"),
                            xytext=(-6, 0), textcoords="offset points",
                            ha="right", va="center", fontsize=7.5, clip_on=False)

        n_label_y = min(row_ys) - 1.0
        ax_dot.text(-0.85, -1.0, "n=", ha="right", va="top", fontsize=5.5, color="#262626")
        for b, (_, ns) in enumerate(block_data):
            base = b * n_trio
            for local_x, ni in zip(range(n_trio), ns):
                ax_dot.text(base + local_x, n_label_y, f"{ni}", ha="center", va="top",
                            fontsize=5.5, color="#262626", clip_on=False)

        ax_dot.set_xlim(-0.7, total_cols - 0.3)
        ax_dot.set_ylim(n_label_y - 1.6, max(row_ys) + 0.6)
        ax_dot.set_xticks([])
        ax_dot.set_yticks([])
        for sp in ax_dot.spines.values():
            sp.set_visible(False)

        ax.set_ylabel(f"{antigen}\n" + r"binding affinity ($-\log_{10}K_D$)", fontsize=11)

    if show_title:
        antigen_note = f" - {antigens[0]}" if len(antigens) == 1 else ""
        restrict_note = f" ({'/'.join(restrict_germline_at)} held germline)" if restrict_germline_at else ""
        fig.suptitle(
            f"{antibody}: higher-order interaction ({', '.join(trio_muts)}) "
            f"x background dependency ({'/'.join(bg_muts)}){antigen_note}{restrict_note}",
            fontsize=14, fontweight="bold", y=1.06,
        )
    plt.tight_layout()

    save_figure(fig, Path(out_name).stem)
    plt.close(fig)

## 6. Make the figures

`drop_or_gate=True` removes the aggregate "I51M OR G56A" block, leaving the four
corners. Both antigens are rendered as separate files sharing one precomputed
`y_range`, so violin heights stay comparable between them (they become panels A
and B of the composed figure).

In [ ]:
df = load_raw_binding_data()

TRIO = ["H35N", "N52K", "S54F"]
BG = ["I51M", "G56A"]
PANEL_ORDER = ["H35N", "N52K", "S54F", "G56A", "I51M"]

# One shared y-range across both antigens, computed before anything is drawn.
y_range = mega_background_dependency_y_range(
    df, "VRC07", TRIO, BG, drop_or_gate=True,
)

for antigen in ANTIGENS:
    mega_background_dependency_figure(
        df, "VRC07", TRIO, BG,
        out_name=f"VRC07_H35N_S54F_N52K_x_I51M_G56A_mega_background_dependency_{antigen}.png",
        antigens=[antigen],
        panel_order=PANEL_ORDER,
        drop_or_gate=True,
        y_range=y_range,
        show_title=False,
        show_panel_titles=False,
    )

### Optional - both antigens side by side in one file

In [ ]:
mega_background_dependency_figure(
    df, "VRC07", TRIO, BG,
    out_name="VRC07_H35N_S54F_N52K_x_I51M_G56A_mega_background_dependency.png",
    panel_order=PANEL_ORDER,
    drop_or_gate=True,
    show_title=False,
    show_panel_titles=False,
)